# 1. Germany monetary policy: source coverage audit

This notebook verifies the retained public source snapshot and checks whether the production, inflation and shock series overlap. No policy responses are estimated and no synthetic observations are generated.

The success criteria are unchanged source hashes, valid date keys and an explicit account of available observations and gaps. A long sample alone does not establish identification or statistical power.

Sources: [project source register](../docs/sources.md) and [data audit](../docs/data_audit.md). The active snapshot was retrieved on 10 October 2026; the September snapshot remains unchanged. Eurostat supplies official indices; the Jarociński update supplies research estimates of ECB monetary-policy and information shocks.

In [1]:
from pathlib import Path
import json

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "scripts/audit_germany_data.py").is_file())
import runpy
from IPython.display import Markdown, display

audit_module = runpy.run_path(str(ROOT / "scripts/audit_germany_data.py"))
report = audit_module["audit"]()
print("Source files passing checksum verification:", report["verified_source_files"])
print(report["interpretation"])


Source files passing checksum verification: 5
Coverage only; no causal estimates, imputation or statistical-power claim.


## 2. Coverage and common sample

The table counts non-missing observations in each source. The final row intersects the three index series and the monthly shocks. Lag construction, transformations, controls and response horizons can subsequently shorten this interval.

In [2]:
table = ["| Series | Observations | First | Last | Internal gaps |",
         "| --- | ---: | --- | --- | ---: |"]
for name, summary in report.items():
    if isinstance(summary, dict) and "observations" in summary:
        gaps = len(summary["missing_within_range"]) if "missing_within_range" in summary else "Event frequency"
        table.append(f"| {name.replace('_', ' ')} | {summary['observations']} | {summary['first']} | {summary['last']} | {gaps} |")
display(Markdown("\n".join(table)))


| Series | Observations | First | Last | Internal gaps |
| --- | ---: | --- | --- | ---: |
| durable production | 428 | 1991-01 | 2026-08 | 0 |
| nondurable production | 428 | 1991-01 | 2026-08 | 0 |
| hicp | 360 | 1996-01 | 2025-12 | 0 |
| monthly shocks | 322 | 1999-01 | 2025-10 | 0 |
| event shocks | 312 | 1999-01-07 | 2025-10-30 | Event frequency |
| common months before transformations | 322 | 1999-01 | 2025-10 | 0 |

## 3. What the indices measure

`MIG_DCOG` and `MIG_NDCOG` are durable and non-durable consumer-goods production groups. They are seasonally and calendar adjusted volume indices with 2021=100. Motor-vehicle manufacturing belongs to capital goods under the official classification. These indices do not measure household consumption.

HICP is the historical, unadjusted German all-items index with 2015=100. A successor classification is available; its use remains a declared design choice. Monthly shocks retain the provider's definitions and normalisation. Recorded zeros remain distinct from missing observations and excluded events.

## 4. Reconciled sources and remaining work

The [source audit](../docs/data_audit.md) reconciles 315 raw announcement dates to 312 retained events through three documented joint-announcement exclusions, verifies monthly aggregation and independently reconstructs the supplied factors and components. The October production vintage adds August 2026 and revises July non-durable production.

I will declare the primary estimand, sample rules, controls and inference before fitting the main model. The snapshot is a retrieval of historical observations, rather than a reconstruction of earlier publication vintages. The 322-month input overlap precedes transformations and horizon losses; it is a coverage result and does not establish the effect of ECB tightening.